<a href="https://colab.research.google.com/github/Wapikkk/text-summarize-nlp/blob/evaluation/notebooks/04_evalutaion.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## **04 - Evaluasi dan Analisis**

**Tujuan:** membandingkan model abstractive (GRU+attention, mT5-small, IndoBART-v2) pada **data test yang sama** dengan satu skrip evaluasi bersama, memakai baseline Lead-2 sebagai titik acuan.

**Masukan:** file prediksi JSONL (`baseline_*.jsonl` dan `abstractive_*.jsonl`) di `results/` dengan format kontrak tim `{"id", "reference", "prediction"}`, serta `test.jsonl` (untuk panjang dokumen dan analisis error).

**Keluaran:** tabel ROUGE-1/2/L (precision, recall, F1) beserta selang kepercayaan, BERTScore, analisis panjang dan waktu, analisis error otomatis, contoh kualitatif, dan sampel anotasi manual. Semua disimpan di folder `evaluation/` di Drive.

**Cara pakai:** jalankan sel dari atas ke bawah. Notebook ini hanya **membaca** hasil, tidak melatih apa pun, sehingga bisa dijalankan ulang kapan saja setelah ada file hasil baru (mis. setelah IndoBART selesai). Model yang file hasilnya belum ada otomatis dilewati. GPU hanya dipakai oleh BERTScore (opsional).

## **Install Package & Import Package**
Memasang `rouge-score` (ROUGE) dan `bert-score` (BERTScore), mengimpor library, dan menghubungkan Google Drive. Kalau tidak di Colab, path Drive tidak dipasang dan notebook memakai path lokal.

In [ ]:
!pip install -q rouge-score bert-score

In [ ]:
import os, re, json, glob, random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from rouge_score import rouge_scorer

from matplotlib.patches import Patch

try:
    import torch
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
except ImportError:
    DEVICE = "cpu"

try:
    from google.colab import drive
    drive.mount("/content/drive")
except ImportError:
    print("Bukan di Colab, memakai path lokal.")

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)
SEED = 42
random.seed(SEED); np.random.seed(SEED)
print("Device:", DEVICE)

## **Konfigurasi**
Lokasi file dan opsi evaluasi. `USE_BERTSCORE = False` melewati BERTScore (butuh mengunduh model dan paling cepat dengan GPU). `BOOTSTRAP_N` adalah jumlah resampling untuk selang kepercayaan 95%. `N_ANNOTATE` adalah jumlah contoh yang disiapkan untuk anotasi error manual.

In [ ]:
DRIVE_DIR  = "/content/drive/MyDrive/dokumen_nlp"
RESULT_DIR = f"{DRIVE_DIR}/results"       # JSONL prediksi semua model (kontrak tim)
TEST_PATH  = f"{DRIVE_DIR}/test.jsonl"    # dokumen asli test (untuk panjang dokumen dan analisis error)
EVAL_DIR   = f"{DRIVE_DIR}/evaluation"    # keluaran evaluasi (CSV dan gambar)
os.makedirs(EVAL_DIR, exist_ok=True)

USE_BERTSCORE = True
BOOTSTRAP_N   = 1000
N_ANNOTATE    = 30
METRICS = {"rouge1": "R1", "rouge2": "R2", "rougeL": "RL"}

## **Memuat Hasil Prediksi**
Membaca `baseline_*.jsonl` dan `abstractive_*.jsonl`, serta catatan waktu (`baseline_run_info.json`, `abstractive_run_info.json`). Hanya `id` yang ada di **semua** model yang dipakai, dan kolom `reference` dicek konsisten, sehingga semua model dinilai pada contoh yang persis sama. Tiap model diberi label yang mudah dibaca (mis. `Lead-2 (baseline)`, `mT5-small`).

In [ ]:
def read_jsonl(path):
    with open(path, encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]

def read_json(path):
    return json.load(open(path, encoding="utf-8")) if os.path.exists(path) else {}

base_info = read_json(f"{RESULT_DIR}/baseline_run_info.json")
abs_info  = read_json(f"{RESULT_DIR}/abstractive_run_info.json")

LABELS = {"baseline_lead_k2": "Lead-2 (baseline)",
          "abstractive_lstm": "GRU+attention",
          "abstractive_mt5_small": "mT5-small",
          "abstractive_indobart_v2": "IndoBART-v2"}
def label(stem):
    return LABELS.get(stem, stem)

files = [p for p in sorted(glob.glob(f"{RESULT_DIR}/*.jsonl"))
         if os.path.basename(p).startswith(("baseline_", "abstractive_"))]
raw = {os.path.basename(p)[:-6]: read_jsonl(p) for p in files}
assert raw, f"Tidak ada file prediksi di {RESULT_DIR}"

# hanya id yang ada di semua model, urutan mengikuti model pertama
first = next(iter(raw))
common = set.intersection(*[{r["id"] for r in rows} for rows in raw.values()])
ids = [r["id"] for r in raw[first] if r["id"] in common]
by_id = {stem: {r["id"]: r for r in rows} for stem, rows in raw.items()}
refs = [by_id[first][i]["reference"] for i in ids]

for stem in raw:
    n_diff = sum(by_id[stem][i]["reference"] != by_id[first][i]["reference"] for i in ids)
    if n_diff:
        print(f"PERINGATAN: {stem} punya {n_diff} reference yang berbeda dari {first}")

test = {r["id"]: r for r in read_jsonl(TEST_PATH)}
docs = [test[i]["document"] if i in test else "" for i in ids]
print("Contoh dengan dokumen:", sum(1 for d in docs if d), "dari", len(ids))

preds_by_model, stem_of, family = {}, {}, {}
for stem in raw:
    name = label(stem)
    preds_by_model[name] = [by_id[stem][i]["prediction"] for i in ids]
    stem_of[name] = stem
    family[name] = "abstractive" if stem.startswith("abstractive_") else "baseline"

print("Model:", list(preds_by_model))
print("Jumlah contoh dievaluasi:", len(ids))

## **Menghitung ROUGE**
ROUGE-1, ROUGE-2, dan ROUGE-L dihitung per contoh dengan `rouge_score` (precision, recall, F1), lalu dirata-rata. Tabel diurutkan menurut ROUGE-1 F1. Karena ringkasan referensi WikiLingua berupa judul langkah yang pendek, ROUGE-2 wajar rendah untuk semua model.

In [ ]:
scorer = rouge_scorer.RougeScorer(list(METRICS), use_stemmer=False)

scores = {}   # model -> {"R1": {"p","r","f"}, ...} (array per contoh)
for name, preds in preds_by_model.items():
    acc = {k: {"p": [], "r": [], "f": []} for k in METRICS.values()}
    for p, r in zip(preds, refs):
        s = scorer.score(r, p)
        for m, short in METRICS.items():
            acc[short]["p"].append(s[m].precision)
            acc[short]["r"].append(s[m].recall)
            acc[short]["f"].append(s[m].fmeasure)
    scores[name] = {k: {x: np.array(v) for x, v in d.items()} for k, d in acc.items()}

rows = []
for name in preds_by_model:
    row = {"model": name, "jenis": family[name]}
    for short in METRICS.values():
        for x, tag in (("p", "P"), ("r", "R"), ("f", "F")):
            row[f"{short}-{tag}"] = scores[name][short][x].mean()
    rows.append(row)
summary = pd.DataFrame(rows).set_index("model").sort_values("R1-F", ascending=False)
summary.round(4)

## **Interval Conivdence dan Uji Perbedaan**
Interval Conivdence 95% dihitung dengan **bootstrap** (resampling contoh test) untuk F1 tiap metrik. Lalu setiap pasangan model dibandingkan lewat **bootstrap berpasangan** pada selisih F1 (misalnya IndoBART vs mT5, mT5 vs GRU, dan tiap model abstractive vs baseline). Kalau selang selisih tidak memuat 0, perbedaannya konsisten. Catatan: beberapa bagian berasal dari artikel yang sama, jadi selang ini sedikit lebih optimistis daripada kenyataan.

In [ ]:
def boot_means(x, n=BOOTSTRAP_N, seed=SEED):
    rng = np.random.default_rng(seed)
    x = np.asarray(x)
    idx = rng.integers(0, len(x), size=(n, len(x)))
    return x[idx].mean(axis=1)

ci_rows = []
for name in summary.index:
    row = {"model": name}
    for short in METRICS.values():
        f = scores[name][short]["f"]
        lo, hi = np.percentile(boot_means(f), [2.5, 97.5])
        row[f"{short}-F1 [95% CI]"] = f"{f.mean():.4f} [{lo:.4f}, {hi:.4f}]"
    ci_rows.append(row)
ci_df = pd.DataFrame(ci_rows).set_index("model")
print(ci_df.to_string())

abs_models = [n for n in summary.index if family[n] == "abstractive"]
base_models = [n for n in summary.index if family[n] == "baseline"]
best_abs = summary.loc[abs_models, "R1-F"].idxmax()
print("Abstractive terbaik:", best_abs, "\n")

pairs = [(a, b) for i, a in enumerate(abs_models) for b in abs_models[i + 1:]] + \
        [(a, b) for a in abs_models for b in base_models]
pair_rows = []
for a, b in pairs:
    row = {"A": a, "B": b}
    for short in METRICS.values():
        d = scores[a][short]["f"] - scores[b][short]["f"]
        lo, hi = np.percentile(boot_means(d), [2.5, 97.5])
        row[f"{short}-F1 selisih (A-B)"] = f"{d.mean():+.4f} [{lo:+.4f}, {hi:+.4f}]"
        if short == "R1":
            row["signifikan (R1)?"] = "ya" if (lo > 0 or hi < 0) else "tidak"
    pair_rows.append(row)
pair_df = pd.DataFrame(pair_rows)
print(pair_df.to_string(index=False))

## **Grafik Perbandingan**
Tiga panel (ROUGE-1, ROUGE-2, ROUGE-L, semua F1) dengan urutan model yang sama (diurutkan menurut ROUGE-1 F1). Warna membedakan jenis: abu-abu untuk baseline, oranye untuk abstractive. Gambar disimpan ke `evaluation/rouge_f1.png`.

In [ ]:
COL_BASE, COL_ABS = "#8f8e89", "#eb6834"          # abu-abu = baseline, oranye = abstractive
INK, INK2, SURFACE, GRID = "#0b0b0b", "#52514e", "#fcfcfb", "#e6e5e1"

order = list(summary.sort_values("R1-F").index)   # terbawah ke teratas (barh)
fig, axes = plt.subplots(1, 3, figsize=(15, 0.5 * len(order) + 2.2), sharey=True, facecolor=SURFACE)
for ax, (col, title) in zip(axes, [("R1-F", "ROUGE-1 F1"), ("R2-F", "ROUGE-2 F1"), ("RL-F", "ROUGE-L F1")]):
    vals = summary.loc[order, col].values
    colors = [COL_ABS if family[n] == "abstractive" else COL_BASE for n in order]
    ax.set_facecolor(SURFACE)
    ax.barh(range(len(order)), vals, height=0.55, color=colors)
    vmax = vals.max()
    for y, v in enumerate(vals):
        ax.text(v + vmax * 0.015, y, f"{v:.3f}", va="center", fontsize=9, color=INK2)
    ax.set_xlim(0, vmax * 1.18)
    ax.set_yticks(range(len(order)))
    ax.set_yticklabels(order, fontsize=10, color=INK)
    ax.set_title(title, fontsize=12, color=INK, loc="left")
    ax.xaxis.grid(True, color=GRID, linewidth=0.8)
    ax.set_axisbelow(True)
    ax.tick_params(axis="x", colors=INK2, labelsize=9, length=0)
    ax.tick_params(axis="y", length=0)
    for sp in ("top", "right", "left"):
        ax.spines[sp].set_visible(False)
    ax.spines["bottom"].set_color(GRID)
fig.legend(handles=[Patch(color=COL_BASE, label="Baseline"), Patch(color=COL_ABS, label="Abstractive")],
           loc="upper right", frameon=False, ncol=2, fontsize=10, labelcolor=INK)
fig.suptitle("Perbandingan ROUGE F1 pada data test", x=0.01, ha="left", fontsize=13, color=INK)
fig.tight_layout(rect=(0, 0, 1, 0.94))
fig.savefig(f"{EVAL_DIR}/rouge_f1.png", dpi=200, facecolor=SURFACE)
plt.show()

## **Analisis Panjang Ringkasan dan Panjang Dokumen**
Dua hal diperiksa. **Pertama**, panjang ringkasan rata-rata tiap model dibanding referensi (ROUGE recall naik kalau ringkasan memanjang, jadi panjang perlu dibandingkan dengan adil). **Kedua**, skor ROUGE-1 F1 dipisah menurut panjang dokumen (pendek, sedang, panjang, dibagi tiga kelompok sama besar) untuk melihat apakah dokumen panjang lebih sulit, terutama bagi model yang memotong input.

In [ ]:
wc = lambda t: len(t.split())
ref_len = np.array([wc(r) for r in refs])
doc_len = np.array([wc(d) for d in docs])

len_rows = []
for name, preds in preds_by_model.items():
    pl = np.array([wc(p) for p in preds])
    len_rows.append({"model": name, "kata/ringkasan": pl.mean(),
                     "rasio thd referensi": pl.mean() / ref_len.mean(),
                     "% prediksi kosong": (pl == 0).mean() * 100})
len_df = pd.DataFrame(len_rows).set_index("model").loc[summary.index]
print(f"Panjang referensi rata-rata: {ref_len.mean():.1f} kata | panjang dokumen rata-rata: {doc_len.mean():.1f} kata\n")
print(len_df.round(2).to_string())

if doc_len.sum() > 0:
    bucket = pd.qcut(pd.Series(doc_len), 3, labels=["pendek", "sedang", "panjang"], duplicates="drop")
    print("\nBatas kelompok panjang dokumen (kata):", pd.Series(doc_len).quantile([1/3, 2/3]).round(0).tolist())
    bucket_df = pd.DataFrame({name: pd.Series(scores[name]["R1"]["f"]).groupby(bucket, observed=True).mean()
                              for name in summary.index}).T
    print("\nROUGE-1 F1 menurut panjang dokumen:")
    print(bucket_df.round(4).to_string())

## **Waktu Training dan Inferensi**
Diambil dari `baseline_run_info.json` dan `abstractive_run_info.json`. Baseline Lead-2 tidak dilatih, sehingga hanya punya waktu inferensi. Kolom yang kosong (`NaN`) berarti catatannya tidak tersimpan (mis. sesi training terputus sehingga `train_time_s` tidak tertulis). Hal ini perlu dicatat jujur di laporan.

In [ ]:
ABS_KEY = {"abstractive_lstm": "lstm", "abstractive_mt5_small": "pretrained", "abstractive_indobart_v2": "indobart"}
time_rows = []
for name in summary.index:
    stem = stem_of[name]
    if stem.startswith("baseline_"):
        d = base_info.get("lead_k2", {})
        train_t = None
    else:
        d = abs_info.get(ABS_KEY.get(stem, ""), {})
        train_t = d.get("train_time_s")
    infer_t, n = d.get("infer_time_s"), d.get("n_test") or abs_info.get("n_test") or len(ids)
    time_rows.append({"model": name, "training (detik)": train_t, "inferensi (detik)": infer_t,
                      "ms/contoh": (infer_t / n * 1000) if infer_t else None})
time_df = pd.DataFrame(time_rows).set_index("model")
time_df.round(2)

## **BERTScore**
Mengukur kemiripan makna dengan embedding BERT multibahasa (`lang="id"` memakai `bert-base-multilingual-cased`). Hasil per model disimpan di `evaluation/bertscore.json` dan dipakai lagi pada run berikutnya, jadi hanya model baru yang dihitung ulang. Hapus file itu kalau hasil prediksi berubah.

In [ ]:
bs_path = f"{EVAL_DIR}/bertscore.json"
bs = read_json(bs_path)
if USE_BERTSCORE:
    from bert_score import score as bert_score
    for name, preds in preds_by_model.items():
        if name in bs:
            continue
        cands = [p if p.strip() else "-" for p in preds]
        P, R, F = bert_score(cands, refs, lang="id", batch_size=32, device=DEVICE, verbose=False)
        bs[name] = {"BS-P": float(P.mean()), "BS-R": float(R.mean()), "BS-F1": float(F.mean())}
        json.dump(bs, open(bs_path, "w"), indent=2)
        print(name, {k: round(v, 4) for k, v in bs[name].items()})
bs_df = pd.DataFrame(bs).T.reindex(summary.index) if bs else pd.DataFrame(index=summary.index)
bs_df.round(4)

## **Analisis Error Otomatis**
Indikator sederhana per model (persentase dari contoh test), berguna sebagai bahan analisis error di laporan:
- **repetisi trigram / kalimat**: ada tiga kata berurutan atau satu kalimat yang muncul berulang di ringkasan.
- **mengandung `<unk>`**: model dari nol menghasilkan token kosakata tidak dikenal (OOV).
- **novelty unigram / bigram**: porsi kata (atau pasangan kata) di ringkasan yang **tidak ada di dokumen**. Baseline Lead-2 hampir 0 karena menyalin kalimat. Nilai tinggi pada abstractive menunjukkan penulisan ulang, tetapi juga bisa menandakan **hallucination**.
- **angka tidak ada di dokumen**: ringkasan memuat angka yang tidak muncul di dokumen (indikator kasar hallucination).

Semua ini hanya **indikator**, bukan bukti. Pemeriksaan manual pada sampel anotasi di bawah tetap diperlukan.

In [ ]:
TOK = re.compile(r"\w+", re.UNICODE)
toks = lambda t: TOK.findall(t.lower())

doc_uni = [set(toks(d)) for d in docs]
doc_bi  = [set(zip(toks(d), toks(d)[1:])) for d in docs]
doc_num = [set(re.findall(r"\d+", d)) for d in docs]

def repeat_flags(p):
    t = toks(p)
    tri = list(zip(t, t[1:], t[2:]))
    sents = [s.strip().lower() for s in re.split(r"(?<=[.!?])\s+", p) if s.strip()]
    return len(tri) != len(set(tri)), len(sents) != len(set(sents))

err_rows = []
for name, preds in preds_by_model.items():
    rep_tri, rep_sent, unk, short, num_bad, nov_u, nov_b = [], [], [], [], [], [], []
    for i, p in enumerate(preds):
        rt, rs = repeat_flags(p)
        rep_tri.append(rt); rep_sent.append(rs)
        unk.append("<unk>" in p.lower())
        t = toks(p)
        short.append(len(t) < 5)
        nums = set(re.findall(r"\d+", p))
        num_bad.append(bool(nums - doc_num[i]))
        if t and docs[i]:
            nov_u.append(np.mean([w not in doc_uni[i] for w in t]))
            bi = list(zip(t, t[1:]))
            nov_b.append(np.mean([b not in doc_bi[i] for b in bi]) if bi else 0.0)
    err_rows.append({"model": name,
                     "% repetisi trigram": np.mean(rep_tri) * 100,
                     "% repetisi kalimat": np.mean(rep_sent) * 100,
                     "% mengandung <unk>": np.mean(unk) * 100,
                     "% terlalu pendek (<5 kata)": np.mean(short) * 100,
                     "% angka tidak ada di dokumen": np.mean(num_bad) * 100,
                     "novelty unigram (%)": (np.mean(nov_u) * 100) if nov_u else np.nan,
                     "novelty bigram (%)": (np.mean(nov_b) * 100) if nov_b else np.nan})
err_df = pd.DataFrame(err_rows).set_index("model").loc[summary.index]
err_df.round(2)

## **Contoh Kualitatif**
Menampilkan beberapa contoh untuk analisis di laporan: tiga contoh **terbaik** dan tiga **terburuk** menurut ROUGE-1 F1 model abstractive terbaik, plus lima contoh acak. Setiap contoh memperlihatkan dokumen (potongan awal), referensi, dan prediksi semua model abstractive serta baseline. Gunakan contoh ini untuk kutipan "benar" dan "salah" di bagian analisis.

In [ ]:
show_models = abs_models + base_models
focus = scores[best_abs]["R1"]["f"]
rank = np.argsort(focus)
rng = np.random.default_rng(SEED)
groups = [("TERBAIK", list(rank[::-1][:3])), ("TERBURUK", list(rank[:3])),
          ("ACAK", list(rng.choice(len(ids), size=5, replace=False)))]

def show(i):
    print("-" * 100)
    print(f"id={ids[i]} | panjang dokumen={doc_len[i]} kata")
    print("DOKUMEN  :", docs[i][:350].replace("\n", " "), "...")
    print("REFERENSI:", refs[i])
    for n in show_models:
        print(f"{n:<28}(R1-F={scores[n]['R1']['f'][i]:.3f}): {preds_by_model[n][i]}")

for title, idxs in groups:
    print("=" * 100); print(title); print("=" * 100)
    for i in idxs:
        show(int(i))

## **Sampel untuk Anotasi Error Manual**
Menyimpan sampel acak ke `evaluation/error_annotation_sample.csv` (buka di Excel/Sheets). Kolom kosong di sisi kanan diisi manual oleh tim dengan 0/1 atau catatan singkat: **halusinasi** (informasi yang tidak ada di dokumen), **repetisi**, **informasi_hilang** (poin penting terlewat), **oov_unk** (kata tidak dikenal), dan **catatan**. Ringkasan jumlahnya dipakai sebagai tabel analisis error di laporan.

In [ ]:
sample_idx = sorted(np.random.default_rng(SEED).choice(len(ids), size=min(N_ANNOTATE, len(ids)), replace=False))
ann = pd.DataFrame({"id": [ids[i] for i in sample_idx],
                    "dokumen (awal)": [docs[i][:400] for i in sample_idx],
                    "referensi": [refs[i] for i in sample_idx]})
for n in abs_models:
    ann[f"prediksi {n}"] = [preds_by_model[n][i] for i in sample_idx]
for c in ["halusinasi", "repetisi", "informasi_hilang", "oov_unk", "catatan"]:
    ann[c] = ""
ann.to_csv(f"{EVAL_DIR}/error_annotation_sample.csv", index=False, encoding="utf-8-sig")
print("Disimpan:", f"{EVAL_DIR}/error_annotation_sample.csv", "| jumlah contoh:", len(ann))

## **Hubungan Antarmodel**
Dua analisis tambahan. **(1) Korelasi Spearman** skor ROUGE-1 F1 per contoh: apakah dua model sama-sama berhasil atau gagal pada contoh yang sama? Korelasi tinggi berarti kesulitan ada pada dokumennya, bukan pada modelnya. **(2) Kemiripan keluaran**: rata-rata ROUGE-1 F1 antara prediksi dua model. Nilai tinggi berarti dua model menghasilkan ringkasan yang mirip. Heatmap disimpan di `evaluation/korelasi_model.png`.

In [ ]:
r1_df = pd.DataFrame({n: scores[n]["R1"]["f"] for n in summary.index})
corr = r1_df.corr(method="spearman")
print("Korelasi Spearman (ROUGE-1 F1 per contoh):")
print(corr.round(2).to_string())

sel = list(summary.index)
sim = pd.DataFrame(1.0, index=sel, columns=sel)
for i, a in enumerate(sel):
    for b in sel[i + 1:]:
        v = np.mean([scorer.score(pa, pb)["rouge1"].fmeasure
                     for pa, pb in zip(preds_by_model[a], preds_by_model[b])])
        sim.loc[a, b] = sim.loc[b, a] = v
print("\nKemiripan keluaran (ROUGE-1 F1 antar prediksi):")
print(sim.round(2).to_string())

def heat(ax, mat, title):
    ax.imshow(mat.values, cmap="Blues", vmin=0, vmax=1)
    ax.set_xticks(range(len(mat))); ax.set_yticks(range(len(mat)))
    ax.set_xticklabels(mat.columns, rotation=45, ha="right", fontsize=9)
    ax.set_yticklabels(mat.index, fontsize=9)
    for i in range(len(mat)):
        for j in range(len(mat)):
            v = mat.values[i, j]
            ax.text(j, i, f"{v:.2f}", ha="center", va="center", fontsize=9,
                    color="white" if (v == v and v > 0.6) else "#0b0b0b")
    ax.set_title(title, fontsize=11, loc="left")

fig, axes = plt.subplots(1, 2, figsize=(12, 5.2), facecolor="#fcfcfb")
heat(axes[0], corr, "Korelasi skor per contoh (Spearman)")
heat(axes[1], sim, "Kemiripan keluaran (ROUGE-1 F1)")
fig.tight_layout()
fig.savefig(f"{EVAL_DIR}/korelasi_model.png", dpi=200, facecolor="#fcfcfb")
plt.show()
corr.to_csv(f"{EVAL_DIR}/korelasi_skor.csv", encoding="utf-8-sig")
sim.to_csv(f"{EVAL_DIR}/kemiripan_keluaran.csv", encoding="utf-8-sig")

## **Menyimpan Tabel untuk Laporan**
Menggabungkan angka utama (ROUGE F1, BERTScore F1, panjang, waktu inferensi) dalam satu tabel, lalu menyimpan semua tabel ke `evaluation/` dalam bentuk CSV. File-file kecil ini boleh diunggah ke GitHub (folder `report/`), sedangkan `bertscore.json` dan gambar opsional.

In [ ]:
main = summary[["jenis", "R1-F", "R2-F", "RL-F"]].copy()
if "BS-F1" in bs_df.columns:
    main["BERTScore-F1"] = bs_df["BS-F1"]
main["kata/ringkasan"] = len_df["kata/ringkasan"]
main["inferensi (detik)"] = time_df["inferensi (detik)"]
main = main.rename(columns={"R1-F": "ROUGE-1 F1", "R2-F": "ROUGE-2 F1", "RL-F": "ROUGE-L F1"})

outputs = {"rouge_summary.csv": summary, "rouge_ci.csv": ci_df, "uji_berpasangan.csv": pair_df, "panjang_ringkasan.csv": len_df,
           "waktu.csv": time_df, "bertscore.csv": bs_df, "analisis_error_otomatis.csv": err_df,
           "tabel_utama.csv": main}
if doc_len.sum() > 0:
    outputs["rouge1_per_panjang_dokumen.csv"] = bucket_df
for fname, df in outputs.items():
    df.to_csv(f"{EVAL_DIR}/{fname}", encoding="utf-8-sig", index=(fname != "uji_berpasangan.csv"))
print("Tersimpan di", EVAL_DIR)
for fname in outputs:
    print(" -", fname)
main.round(4)

## **Catatan untuk Laporan**
- **Metrik resmi** adalah ROUGE-1/2/L F1 dari sel di atas. Tuliskan juga precision dan recall (ada di `rouge_summary.csv`), BERTScore, selang kepercayaan, uji berpasangan, serta waktu.
- **Baseline:** Lead-2 (dua kalimat pertama dokumen, rata-rata sekitar 30 kata, setara panjang referensi) adalah titik acuan, bukan model yang dilatih. Model abstractive perlu mengungguli skornya agar dianggap benar-benar belajar meringkas.
- **Pertanyaan utama:** apakah pretraining membantu (GRU dari nol vs mT5 dan IndoBART), dan apakah pretraining khusus bahasa Indonesia (IndoBART) lebih baik daripada pretraining multibahasa (mT5)? Jawab dengan tabel uji berpasangan.
- **Keterbatasan** yang perlu disebut: dokumen dipotong (GRU 300 kata, mT5 dan IndoBART 512 token); referensi berupa judul langkah sehingga ROUGE-2 rendah; GRU dilatih dari nol sehingga rentan OOV dan repetisi; tokenizer dan ukuran model ketiganya berbeda sehingga selisih skor bukan murni karena jenis pretraining; catatan epoch dan waktu training GRU dan mT5 tidak lengkap (lihat tabel waktu).
- **Analisis error:** pakai tabel otomatis sebagai petunjuk, lalu isi `error_annotation_sample.csv` secara manual untuk angka yang bisa dipertanggungjawabkan.
- **Setelah selesai:** unggah file CSV dan PNG di `evaluation/` ke folder `report/` di branch `evaluation` GitHub, dan simpan notebook ini sebagai `notebooks/04_evaluation.ipynb`.